## 1. Dataset  

At a high level, the Numerai dataset is a tabular dataset that describes the stock market over time. It is compiled from high-quality (and expensive) data that might be difficult for individuals to obtain.

The unique thing about Numerai's dataset is that it is `obfuscated`, which means that the underlying stock ids, feature names, and target definitions are anonymized. This makes it so that we can give this data out for free and so that it can be modeled without any financial domain knowledge (or bias!).
### Listing the datasets
Firstly, take a look at the files Numerai offers below:

In [1]:
# Initialize NumerAPI - the official Python API client for Numerai
from numerapi import NumerAPI
napi = NumerAPI()

In [2]:
# list the datasets and available versions
all_datasets = napi.list_datasets()
dataset_versions = list(set(d.split('/')[0] for d in all_datasets))
assert len(dataset_versions) > 0, "No dataset versions found."
assert len(dataset_versions) == 3, "[WARNING] Expected 3 dataset versions, but found {}. Please check the Numerai API for updates.".format(len(dataset_versions))
print("Available versions:\n", dataset_versions)

Available versions:
 ['v5.0', 'v5.2', 'v5.1']


In [3]:
# Set data version to one of the latest datasets
DATA_VERSION = "v5.2"

# Print all files available for download for our version
current_version_files = [f for f in all_datasets if f.startswith(DATA_VERSION)]
print("Available", DATA_VERSION, "files:\n", current_version_files)
# Ensure HTTPS certificate verification uses an up-to-date CA bundle (avoids InsecureRequestWarning)
import os, certifi
os.environ["SSL_CERT_FILE"] = certifi.where()
os.environ["REQUESTS_CA_BUNDLE"] = certifi.where()

Available v5.2 files:
 ['v5.2/features.json', 'v5.2/live.parquet', 'v5.2/live_benchmark_models.parquet', 'v5.2/live_example_preds.csv', 'v5.2/live_example_preds.parquet', 'v5.2/meta_model.parquet', 'v5.2/train.parquet', 'v5.2/train_benchmark_models.parquet', 'v5.2/validation.parquet', 'v5.2/validation_benchmark_models.parquet', 'v5.2/validation_example_preds.csv', 'v5.2/validation_example_preds.parquet']


 ['v5.2/features.json', 'v5.2/live.parquet', 'v5.2/live_benchmark_models.parquet', 'v5.2/live_example_preds.csv', 'v5.2/live_example_preds.parquet', 'v5.2/meta_model.parquet', 'v5.2/train.parquet', 'v5.2/train_benchmark_models.parquet', 'v5.2/validation.parquet', 'v5.2/validation_benchmark_models.parquet', 'v5.2/validation_example_preds.csv', 'v5.2/validation_example_preds.parquet']

### Downloading datasets

The `features.json` file contains metadata about features in the dataset including:
- statistics on each feature
- helpful sets of features
- the targets available for training

In [4]:
import json

# download the feature metadata file
napi.download_dataset(f"{DATA_VERSION}/features.json")

# read the metadata and display
feature_metadata = json.load(open(f"{DATA_VERSION}/features.json"))
for metadata in feature_metadata:
  print(metadata, len(feature_metadata[metadata]))

2026-06-20 20:07:15,834 INFO numerapi.utils: target file already exists
2026-06-20 20:07:15,834 INFO numerapi.utils: download complete


feature_sets 18
targets 41


In [5]:
# download the feature metadata file
napi.download_dataset(f"{DATA_VERSION}/features.json")

# read the metadata and display
feature_metadata = json.load(open(f"{DATA_VERSION}/features.json"))
for metadata in feature_metadata:
  print(metadata, len(feature_metadata[metadata]))

2026-06-20 20:07:21,217 INFO numerapi.utils: target file already exists
2026-06-20 20:07:21,219 INFO numerapi.utils: download complete


feature_sets 18
targets 41


### Feature Sets & Groups
As you can see there are many features and targets to choose from.

Instead of training a model on all 2000+ features, let's pick a subset of features to analyze.

Here are a few starter sets Numerai offers:

- `small` contains a minimal subset of features that have the highest [feature importance](https://scikit-learn.org/stable/auto_examples/ensemble/plot_forest_importances.html)

- `medium` contains all the "basic" features, each unique in some way (e.g. P/E ratios vs analyst ratings)

- `all` contains all features in `medium` and their variants (e.g. P/E by country vs P/E by sector)

In [6]:
feature_sets = feature_metadata["feature_sets"]
for feature_set in ["small", "medium", "all"]:
  print(feature_set, len(feature_sets[feature_set]))

small 42
medium 780
all 2748


---
# Data Refresh — Round-Based Strategy

Numerai runs a new tournament round **Tuesday–Saturday**. The datasets have different refresh frequencies:

| Dataset | Refresh Frequency | Priority |
|---|---|---|
| `live.parquet` | **Every round (daily)** | 🔴 Critical |
| `live_benchmark_models.parquet` | Every round | High |
| `train.parquet` / `validation.parquet` | New data version only | Low |
| `features.json` | New data version only | Low |

The script below uses `napi.get_current_round()` to check whether the API's current round matches the last round recorded in your local CSV. It will force-download `live.parquet` only when the round has advanced (or the file is missing), then logs the `round_id` alongside era metadata.


In [7]:
import os
import json
import pandas as pd
from datetime import date

# --- Configuration ---
current_data_dir = 'C:\\dev\\numer-AI\\data'
era_csv_path = f"{current_data_dir}/numerai_era_data.csv"
live_file_path = f"{current_data_dir}/{DATA_VERSION}/live.parquet"

# --- 1. Identify the Current Live Round ---
current_round = napi.get_current_round()
print(f"Current Tournament Round: {current_round}")

# --- 2. Check Local Records ---
last_recorded_round = None
try:
    df_existing = pd.read_csv(era_csv_path)
    live_records = df_existing[df_existing["dataset"] == "live"]
    if not live_records.empty and "round_id" in live_records.columns:
        last_recorded_round = live_records["round_id"].max()
except (FileNotFoundError, pd.errors.EmptyDataError, KeyError):
    df_existing = pd.DataFrame()

print(f"Last Downloaded Round : {last_recorded_round}")

# --- 3. Refresh Strategy ---
# Force download if the round has advanced OR if live.parquet is missing
needs_refresh = (current_round != last_recorded_round) or (not os.path.exists(live_file_path))

if needs_refresh:
    print(f"\nRefreshing live data for Round {current_round}...")
    napi.download_dataset(f"{DATA_VERSION}/live.parquet", dest_path=live_file_path)

    # Also ensure non-live files are present (download if missing)
    static_files = [
        f"{DATA_VERSION}/train.parquet",
        f"{DATA_VERSION}/validation.parquet",
        f"{DATA_VERSION}/features.json",
    ]
    for f in static_files:
        dest = f"{current_data_dir}/{f}"
        if not os.path.exists(dest):
            print(f"  Downloading missing static file: {f}")
            napi.download_dataset(f, dest_path=dest)

    # --- 4. Update Era CSV with Round Metadata ---
    today = str(date.today())
    main_datasets = {
        "train":      f"{current_data_dir}/{DATA_VERSION}/train.parquet",
        "validation": f"{current_data_dir}/{DATA_VERSION}/validation.parquet",
        "live":       live_file_path,
    }

    new_records = []
    for name, path in main_datasets.items():
        if os.path.exists(path):
            df_tmp = pd.read_parquet(path, columns=["era"])
            new_records.append({
                "date":      today,
                "dataset":   name,
                "round_id":  current_round if name == "live" else None,
                "start_era": df_tmp["era"].min(),
                "end_era":   df_tmp["era"].max(),
            })

    df_new = pd.DataFrame(new_records)

    # Remove today's stale entries then append fresh rows
    if not df_existing.empty:
        df_existing = df_existing[df_existing["date"] != today]
        df_combined = pd.concat([df_existing, df_new], ignore_index=True)
    else:
        df_combined = df_new

    df_combined.to_csv(era_csv_path, index=False)
    print(f"\nRecords updated for Round {current_round}.")
else:
    print("Live data is already up to date for the current round.")
    df_combined = df_existing

# --- 5. Display Final Status ---
print("\nLatest era records:")
print(pd.read_csv(era_csv_path).tail(6).to_string(index=False))


Current Tournament Round: 1294
Last Downloaded Round : 1211.0

Refreshing live data for Round 1294...


2026-06-20 20:07:26,652 INFO numerapi.utils: target file already exists
2026-06-20 20:07:26,652 INFO numerapi.utils: starting download
C:\dev\numer-AI\data/v5.2/live.parquet: 19.6MB [00:01, 12.1MB/s]                   



Records updated for Round 1294.

Latest era records:
      date    dataset start_era end_era  round_id
2026-02-25      train      0001    0574       NaN
2026-02-25 validation      0575    1208       NaN
2026-02-25       live         X       X    1211.0
2026-06-20      train      0001    0574       NaN
2026-06-20 validation      0575    1208       NaN
2026-06-20       live         X       X    1294.0


In [8]:
df_live = pd.read_parquet(live_file_path, columns=["era"])
print(f"Live data: {len(df_live):,} rows  |  Eras: {df_live['era'].min()} → {df_live['era'].max()}")
df_live


Live data: 7,076 rows  |  Eras: X → X


,era
id,
n0004edbadf9559f,X
n00080ede54e02fe,X
n000a29996609c7a,X
n0016991ea828483,X
n002bf0209f33d26,X
...,...
nffe707495ce1edc,X
nffe8bd3e8aac007,X
nffece480c78128e,X
